# Chapter 8 &mdash; Regular Expressions: Syntax, Denotation, Precedence

**Concept 1 of the Chapter 8 decomposition:** *Regular Expressions: Syntax, Denotation, and Precedence*

Primitives $\varepsilon$, $a$, $\emptyset$ glued by union, concatenation and star &mdash; with star tightest and union loosest.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8/Concept-RE-Syntax-And-Denotation/Concept-RE-Syntax-And-Denotation.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


A regular expression is built from three primitives &mdash; $\varepsilon$, a single symbol
$a$, and $\emptyset$ &mdash; using three operators:

* **union** $R_1 + R_2$,
* **concatenation** $R_1 R_2$ (written by juxtaposition),
* **star** $R^*$.

**Precedence: star binds tightest, then concatenation, then union.** So $01^*$ is
$0(1^*)$, not $(01)^*$, and $0+10$ is $0 + (10)$.

The **denotation** $L(R)$ is a language. Two different expressions can denote the same
language, which is why "are these two REs equal?" is a real question &mdash; answered by
converting both to minimal DFA.

In Jove, `''` or `""` denotes $\varepsilon$, and symbols are single alphanumerics.

## 2. Definitions

### Building machines from expressions

In [ ]:
def re_dfa(r): return min_dfa(nfa2dfa(re2nfa(r)))

def lang(r, n=8, sigma=None):
    # Strings of L(r) up to length n.  Enumerate over the MACHINE's own
    # alphabet -- L('') has an empty Sigma, and step_dfa asserts on a
    # symbol outside Sigma.
    from itertools import product
    D = re_dfa(r)
    sig = sorted(D["Sigma"]) if sigma is None else list(sigma)
    return [''.join(p) for k in range(n+1) for p in product(sig, repeat=k)
            if accepts_dfa(D, ''.join(p))]

### The three primitives

In [ ]:
eps  = re2nfa("''")          # epsilon
sym  = re2nfa("0")           # a single symbol
print("epsilon NFA states :", len(eps["Q"]))
print("symbol  NFA states :", len(sym["Q"]))

## 3. Tests

The primitives denote exactly what they should.

In [ ]:
print("L('')  up to length 3 :", lang("''", 3))
print("L('0') up to length 3 :", lang("0", 3))
assert lang("''", 3) == ['']
assert lang("0", 3) == ['0']

**Precedence: star binds tightest.** $01^*$ and $(01)^*$ are different languages.

In [ ]:
a = lang("01*", 4)
b = lang("(01)*", 4)
print("L(01*)   :", a)
print("L((01)*) :", b)
assert a != b
assert '0111' in a and '0111' not in b
assert '0101' in b and '0101' not in a

**Concatenation binds tighter than union.** $0+10 \ne (0+1)0$.

In [ ]:
print("L(0+10)  :", lang("0+10", 3))
print("L((0+1)0):", lang("(0+1)0", 3))
assert re_dfa("0+10") and not langeq_dfa(re_dfa("0+10"), re_dfa("(0+1)0"))

Different expressions, same language &mdash; decided by minimal DFA.

In [ ]:
pairs = [("(0+1)*", "(0*1*)*"), ("0*0*", "0*"), ("(01)*0", "0(10)*")]
for r1, r2 in pairs:
    same = langeq_dfa(re_dfa(r1), re_dfa(r2))
    print("%-12s == %-12s ? %s" % (r1, r2, same))
    assert same

And a pair that only *looks* equal.

In [ ]:
print("(0+1)* == 0*+1* ?", langeq_dfa(re_dfa("(0+1)*"), re_dfa("0*+1*")))
assert not langeq_dfa(re_dfa("(0+1)*"), re_dfa("0*+1*"))
print("witness : '01' is in the first, not the second ->",
      accepts_dfa(re_dfa("(0+1)*"), '01'), accepts_dfa(re_dfa("0*+1*"), '01'))

## 4. Animation

The minimal DFA for $(0+1)^*1(0+1)(0+1)$ &mdash; an RE, turned into a machine.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(re_dfa('(0+1)*1(0+1)(0+1)'), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Parenthesise $0+1^*0^*$ fully according to the precedence rules.
2. Is $\emptyset^* = \varepsilon$? Argue it from the definition of star.
3. Find two REs of different length denoting the same language, and prove it with `langeq_dfa`.

In [ ]:
# Your work for the exercises above.